In [ ]:
# run_pc_with_ols_weights.py
# ------------------------------------------------------------
# PC algorithm (causal-learn) -> directed edges only (i->j 확정된 것만)
# + OLS regression to estimate edge weights on the fixed PC structure
#
# Requirements:
#   pip install numpy pandas networkx causal-learn
# ------------------------------------------------------------

import os
import json
import warnings
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# =========================
# Config
# =========================
DATA_PATH = "./training_data_standardization.csv"
OUT_BASE = "./dag_out/PC"
RANDOM_STATE = 42
MAX_FEATURES = None

TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]

# PC
ALPHA = 0.01

# OLS weight estimation config (✅ PC 구조 고정 후 계수 추정)
USE_STANDARDIZE_FOR_OLS = True   # NOTEARS/GOLEM과 비교용으로 권장
OLS_RCOND = None                # np.linalg.lstsq rcond

# "0 edge" 제거 기준 (저장/그래프에서 제외)
EPS_EDGE = 1e-12  # 진짜 0만 제거하려면 0.0, 사실상 0도 제거하려면 1e-6 등


# =========================
# Data
# =========================
def load_numeric_X(
    data_path: str,
    drop_target_candidates: bool = True,
    max_features: Optional[int] = None,
    random_state: int = 42
) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    np.random.seed(random_state)
    df = pd.read_csv(data_path, low_memory=False)

    # 흔한 인덱스 컬럼 제거(있으면)
    for c in ["Unnamed: 0", "index", "__index_level_0__"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    if drop_target_candidates:
        # 대소문자/공백 변형 흡수
        norm_cols = {c.strip().lower(): c for c in df.columns}
        drop_cols = []
        for tc in TARGET_CANDIDATES:
            if tc in norm_cols:
                drop_cols.append(norm_cols[tc])
        if drop_cols:
            print(f"[INFO] drop target candidates: {drop_cols}")
            df = df.drop(columns=drop_cols)

    # numeric only
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    df = df[num_cols].copy()

    if max_features is not None and df.shape[1] > max_features:
        df = df.iloc[:, :max_features].copy()
        print(f"[INFO] feature capped: {max_features}")

    # NaN -> median
    for c in df.columns:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    col_names = df.columns.tolist()
    X = df.values.astype(float)

    # PC fisher-z에서는 mean-center로 충분 (표준화는 독립성검정엔 크게 영향 적지만 일관성 위해 센터링 유지)
    X = X - X.mean(axis=0, keepdims=True)

    print(f"[INFO] X shape: {X.shape}")
    return df, X, col_names


# =========================
# Graph helpers
# =========================
def extract_directed_edges_from_causallearn_graph(G: Any, d: int) -> List[Tuple[int, int]]:
    """
    causal-learn Graph 객체에서 i->j 방향 간선을 추출.
    CPDAG의 경우 방향 확정된 i->j만 추출.
    """
    mat = getattr(G, "graph", None)
    if mat is None:
        raise ValueError("Cannot find adjacency matrix 'G.graph' in causal-learn Graph object.")

    mat = np.asarray(mat)
    if mat.shape != (d, d):
        raise ValueError(f"G.graph shape mismatch: {mat.shape} vs d={d}")

    edges: List[Tuple[int, int]] = []

    # Case A: i->j : mat[i,j]==1 and mat[j,i]==-1
    for i in range(d):
        for j in range(d):
            if i != j and mat[i, j] == 1 and mat[j, i] == -1:
                edges.append((i, j))
    if edges:
        return list(dict.fromkeys(edges))

    # Case B: i->j : mat[i,j]==-1 and mat[j,i]==1  (버전/표현 차이 대비)
    for i in range(d):
        for j in range(d):
            if i != j and mat[i, j] == -1 and mat[j, i] == 1:
                edges.append((i, j))
    if edges:
        return list(dict.fromkeys(edges))

    # Case C: mat[i,j]!=0 and mat[j,i]==0 를 i->j로 (fallback)
    for i in range(d):
        for j in range(d):
            if i != j and mat[i, j] != 0 and mat[j, i] == 0:
                edges.append((i, j))

    return list(dict.fromkeys(edges))


def break_cycles_by_removing_small_edges(W: np.ndarray) -> np.ndarray:
    """
    cycle이 있으면, cycle에 포함된 edge 중 |W|가 가장 작은 edge 제거.
    (PC는 원래 cycle-free여야 하는데, 후처리/표현 문제 대비 안전장치)
    """
    W2 = W.copy()

    def build_graph(Wm):
        adj = {i: [] for i in range(Wm.shape[0])}
        for i in range(Wm.shape[0]):
            for j in range(Wm.shape[1]):
                if i != j and abs(Wm[i, j]) > 0:
                    adj[i].append(j)
        return adj

    def find_cycle_edges(adj):
        d = len(adj)
        color = [0] * d
        parent = [-1] * d

        def dfs(u):
            color[u] = 1
            for v in adj[u]:
                if color[v] == 0:
                    parent[v] = u
                    cyc = dfs(v)
                    if cyc is not None:
                        return cyc
                elif color[v] == 1:
                    nodes = [v]
                    cur = u
                    while cur != v and cur != -1:
                        nodes.append(cur)
                        cur = parent[cur]
                    nodes.append(v)
                    nodes = nodes[::-1]
                    return [(a, b) for a, b in zip(nodes[:-1], nodes[1:])]
            color[u] = 2
            return None

        for s in range(d):
            if color[s] == 0:
                cyc = dfs(s)
                if cyc is not None:
                    return cyc
        return None

    while True:
        cyc = find_cycle_edges(build_graph(W2))
        if cyc is None:
            break

        mags = [(abs(W2[i, j]), i, j) for (i, j) in cyc]
        mags.sort(key=lambda x: x[0])
        _, i_min, j_min = mags[0]
        W2[i_min, j_min] = 0.0

    return W2


# =========================
# OLS weights on fixed structure
# =========================
def standardize(X: np.ndarray) -> np.ndarray:
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True)
    sd = np.where(sd < 1e-12, 1.0, sd)
    return (X - mu) / sd


def compute_ols_weights_for_edges(
    X: np.ndarray,
    edges: List[Tuple[int, int]],
    d: int,
    standardize_for_ols: bool = True
) -> np.ndarray:
    """
    PC로 얻은 구조(부모집합)를 고정하고,
    각 노드 j에 대해: X_j ~ X_{Pa(j)} 선형회귀(절편 없음)로 계수 추정.
    반환 W_beta (d x d): i->j일 때 W_beta[i,j] = beta_{i->j}
    """
    if len(edges) == 0:
        return np.zeros((d, d), dtype=float)

    Xr = standardize(X) if standardize_for_ols else X.copy()

    parents_of: Dict[int, List[int]] = {j: [] for j in range(d)}
    for i, j in edges:
        parents_of[j].append(i)

    W_beta = np.zeros((d, d), dtype=float)

    for j in range(d):
        pa = sorted(list(dict.fromkeys(parents_of[j])))
        if len(pa) == 0:
            continue

        y = Xr[:, j]
        Xp = Xr[:, pa]

        coef, *_ = np.linalg.lstsq(Xp, y, rcond=OLS_RCOND)
        coef = coef.astype(float)

        for k, i in enumerate(pa):
            W_beta[i, j] = float(coef[k])

    return W_beta


# =========================
# Save artifacts (weight only)
# =========================
def save_artifacts(
    W_beta: np.ndarray,
    col_names: List[str],
    out_dir: str,
    base_name: str,
    eps_edge: float = 0.0
) -> None:
    import networkx as nx

    os.makedirs(out_dir, exist_ok=True)

    d = len(col_names)
    rows = []
    for i in range(d):
        for j in range(d):
            if i == j:
                continue
            beta = float(W_beta[i, j])
            if abs(beta) <= eps_edge:
                continue
            rows.append([col_names[i], col_names[j], beta])

    edge_df = pd.DataFrame(rows, columns=["source", "target", "weight"])
    edge_path = os.path.join(out_dir, f"edges_{base_name}.csv")
    edge_df.to_csv(edge_path, index=False)

    adj_beta = pd.DataFrame(W_beta, index=col_names, columns=col_names)
    beta_path = os.path.join(out_dir, f"adj_{base_name}_weight.csv")
    adj_beta.to_csv(beta_path)

    # Graph (weight)
    G = nx.DiGraph()
    for n in col_names:
        G.add_node(n)
    for _, r in edge_df.iterrows():
        G.add_edge(r["source"], r["target"], weight=float(r["weight"]))

    gexf_path = os.path.join(out_dir, f"graph_{base_name}.gexf")
    graphml_path = os.path.join(out_dir, f"graph_{base_name}.graphml")
    nx.write_gexf(G, gexf_path)
    nx.write_graphml(G, graphml_path)

    nodes = [{"id": n} for n in col_names]
    jedges = [
        {"source": r["source"], "target": r["target"], "weight": float(r["weight"])}
        for _, r in edge_df.iterrows()
    ]
    json_path = os.path.join(out_dir, f"graph_{base_name}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"nodes": nodes, "edges": jedges}, f, ensure_ascii=False, indent=2)

    print(f"[SAVE] {base_name}")
    print(f"  - {edge_path} (n_edges={len(edge_df)})")
    print(f"  - {beta_path}")
    print(f"  - {graphml_path}")
    print(f"  - {gexf_path}")
    print(f"  - {json_path}")
    print(f"  - filtered: abs(weight) > {eps_edge}")


# =========================
# Main
# =========================
def main():
    _, X, col_names = load_numeric_X(
        DATA_PATH,
        drop_target_candidates=True,
        max_features=MAX_FEATURES,
        random_state=RANDOM_STATE
    )

    try:
        from causallearn.search.ConstraintBased.PC import pc
        from causallearn.utils.cit import fisherz
    except Exception as e:
        raise ImportError(
            "PC 실행을 위해 causal-learn이 필요합니다.\n"
            "pip install causal-learn\n"
            f"Original error: {e}"
        )

    # 1) Run PC (CPDAG)
    cg = pc(X, alpha=ALPHA, indep_test=fisherz, stable=True, uc_rule=0, uc_priority=2)
    G = cg.G

    d = len(col_names)

    # 2) Extract directed edges (only oriented i->j)
    edges_ij = extract_directed_edges_from_causallearn_graph(G, d)
    print(f"[INFO] directed edges from PC: {len(edges_ij)}")

    # 3) Fit OLS coefficients on the fixed PC structure (edge weights)
    W_beta = compute_ols_weights_for_edges(
        X=X,
        edges=edges_ij,
        d=d,
        standardize_for_ols=USE_STANDARDIZE_FOR_OLS
    )

    # (선택) 안전 DAG enforcement:
    # OLS는 구조를 바꾸지 않지만, 저장할 때 eps_edge로 0 근처 제거하면
    # 그래프 표현상 사이클이 생길 일은 거의 없습니다. 그래도 혹시 대비하려면:
    W_beta = break_cycles_by_removing_small_edges(W_beta)

    # 4) Save artifacts
    save_artifacts(
        W_beta=W_beta,
        col_names=col_names,
        out_dir=OUT_BASE,
        base_name="PC",
        eps_edge=EPS_EDGE
    )


if __name__ == "__main__":
    main()


[INFO] drop target candidates: ['label']
[INFO] X shape: (17881, 13)


Depth=6, working on node 12: 100%|██████████| 13/13 [00:00<00:00, 375.50it/s]


[INFO] directed edges from PC: 26
[SAVE] PC
  - ./dag_out/PC\edges_PC.csv (n_edges=24)
  - ./dag_out/PC\adj_PC_weight.csv
  - ./dag_out/PC\graph_PC.graphml
  - ./dag_out/PC\graph_PC.gexf
  - ./dag_out/PC\graph_PC.json
  - filtered: abs(weight) > 1e-12
